# V-UNIVERSAL — demo notebook

Cleaned, reproducible companion to the `v_universal` package. Each section mirrors one of the scripts under `examples/`.

Repository: https://github.com/your-org/v-universal  
License: Apache-2.0

In [ ]:
from v_universal import (
    VHardware, VQuantum, VBio, VPhysics, VMath, VLaw,
    VEconomy, VConsciousness, bench_universal,
)
from v_universal.economy import InsufficientFunds
from v_universal.benchmark import format_results

## 1. V-HARDWARE — hardware transactional memory

In [ ]:
hw = VHardware()
hw.begin_tx()
hw.write_reg(0, 42)
hw.write_reg(1, 100)
hw.bus_write(0x1000, 0xDEAD)
print(f'in tx:    r0={hw.read_reg(0)}, bus.addr=0x{hw.tx["bus"].read("addr"):X}')
hw.abort_tx()
print(f'after abort: trunk r0={hw.regs.get("r0")}, bus.addr=0x{hw.bus.get("addr"):X}')
hw.begin_tx()
hw.write_reg(0, 42)
hw.commit_tx()
print(f'after commit: trunk r0={hw.regs.get("r0")}')

## 2. V-QUANTUM — state vector as a versioned trunk

In [ ]:
import numpy as np
q = VQuantum(n_qubits=4)
v = q.state.fork()
q.apply_gate(v, q.hadamard(), 0)
q.apply_gate(v, q.hadamard(), 1)
nonzero = np.nonzero(np.abs(v.read('vec'))**2 > 1e-9)[0]
print(f'after H on qubits 0,1: {len(nonzero)} non-zero states')
outcome = q.measure(v, rng=np.random.default_rng(0))
print(f'measured: |{outcome:04b}>')
v.commit()
print(f'committed: |{int(np.argmax(np.abs(q.state.get("vec")))):04b}>')

## 3. V-BIO — genome + mutations + selection

In [ ]:
bio = VBio(genome_size=64, seed=42)
print(f'base genome: {"".join(bio.genome.get("seq")[:16])}...')
pop = []
for _ in range(10):
    v = bio.genome.fork()
    bio.mutate(v, rate=0.1)
    bio.evaluate(v)
    pop.append(v)
best = bio.select(pop, k=1)[0]
print(f'best fitness: {best.read("fitness"):.3f}')
best.commit()
print(f'committed: {"".join(bio.genome.get("seq")[:16])}...')

## 4. V-PHYSICS — multiverse with different gravity values

In [ ]:
phy = VPhysics()
for G, name in [(1.62, 'Moon'), (9.81, 'Earth'), (24.79, 'Jupiter')]:
    v = phy.world.fork()
    v.write('G', G)
    phy.add_body(v, (0, 10), (1, 0))
    phy.run(v, 100)
    b = v.read('bodies')[0]
    print(f'{name:8s} G={G:5.2f}: pos=({b["pos"][0]:.2f}, {b["pos"][1]:.2f})')
    v.abort()

## 5. V-MATH — axioms, theorems, failed proofs

In [ ]:
m = VMath()
v1 = m.axioms.fork()
m.prove(v1, 'a + 0 = a', ['add: a + 0 = a'])
v1.commit()
print(f'committed theorems: {list(m.axioms.get("theorems").keys())}')
v2 = m.axioms.fork()
m.prove(v2, '1 = 2', ['faulty'])
v2.abort()
print(f'after abort: {list(m.axioms.get("theorems").keys())}')

## 6. V-LAW — statutes and constitutional amendments

In [ ]:
law = VLaw()
v = law.const.fork()
law.propose_law(v, 'law_1', 'Flat tax 13%')
law.amend_constitution(v, 'art4', 'Right to education')
v.commit()
print(f'articles: {list(law.articles().keys())}')
print(f'laws: {list(law.const.get("laws").keys())}')

## 7. V-ECONOMY — ledger, transfers, parallel branches

In [ ]:
eco = VEconomy()
v1 = eco.ledger.fork()
eco.transfer(v1, 'alice', 'bob', 100)
v1.commit()
print(f'after tx1: {eco.ledger.get("accounts")}')
v2 = eco.ledger.fork()
try:
    eco.transfer(v2, 'carol', 'alice', 10000)
    v2.commit()
except InsufficientFunds as e:
    print(f'tx failed: {e}')
    v2.abort()
print(f'after failed tx: {eco.ledger.get("accounts")}')

## 8. V-CONSCIOUSNESS — thoughts, decisions, regret

In [ ]:
m = VConsciousness()
v1 = m.mind.fork()
m.think(v1, 'I see light')
m.decide(v1, 'Open eyes')
v1.commit()
print(f'thoughts: {m.thoughts()}, decision: {m.decision()!r}')
v2 = m.mind.fork()
m.think(v2, 'I want to scream')
m.decide(v2, 'Scream')
v2.abort()  # regret
print(f'after regret: decision = {m.decision()!r}')

## 9. Cross-domain benchmark

In [ ]:
results = bench_universal(n=10000)
print(format_results(results, n=10000))